<a href="https://colab.research.google.com/github/YolandaZhao10/CSCI-6170-Project-in-AI-and-ML/blob/main/lab02/Lab02_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#2 Data sanity check
import numpy as np
import os

# Data Sanity Checks
def load_split(split_path):
    signals = [
        "body_acc_x", "body_acc_y", "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
        "total_acc_x", "total_acc_y", "total_acc_z"
    ]

    data_list = []

    for sig in signals:
        file_path = os.path.join(f"{sig}_{os.path.basename(split_path)}.txt")
        data = np.loadtxt(file_path)
        data_list.append(data)

    # Stack as (num_windows, T, C)
    X = np.stack(data_list, axis=-1)
    return X
X_train = load_split("train")
X_test = load_split("test")
y_train = np.loadtxt("y_train.txt")
y_test = np.loadtxt("y_test.txt")

FileNotFoundError: body_acc_x_train.txt not found.

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Model
# --------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, T, C)
        lstm_out, _ = self.lstm(x)

        # use the last timestep output
        last_hidden = lstm_out[:, -1, :]   # shape: (batch, hidden_dim)

        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits


model = LSTMClassifier(
    input_dim=9,
    hidden_dim=128,
    num_layers=2,
    num_classes=6,
    dropout=0.3
).to(device)

# --------------------------------------------------
# Example: move tensors to device
# --------------------------------------------------
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_val   = torch.tensor(X_test, dtype=torch.float32).to(device)
y_val   = torch.tensor(y_test, dtype=torch.long).to(device)
X_test  = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test, dtype=torch.long).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# Reduce LR when validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# --------------------------------------------------
# Training settings
# --------------------------------------------------
epochs = 50
max_grad_norm = 1.0      # gradient clipping threshold
early_stop_patience = 7

best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

for epoch in range(epochs):
    # -------------------------
    # Train
    # -------------------------
    model.train()
    optimizer.zero_grad()

    train_logits = model(X_train)
    train_loss = criterion(train_logits, y_train)

    train_loss.backward()

    # Gradient clipping for RNN/LSTM robustness
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    optimizer.step()

    train_preds = torch.argmax(train_logits, dim=1)
    train_acc = (train_preds == y_train).float().mean().item()

    # -------------------------
    # Validate
    # -------------------------
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val)

        val_preds = torch.argmax(val_logits, dim=1)
        val_acc = (val_preds == y_val).float().mean().item()

    # Step scheduler on validation loss
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss.item():.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss.item():.4f} | Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # -------------------------
    # Early stopping
    # -------------------------
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

# Load best model
model.load_state_dict(best_model_wts)

# --------------------------------------------------
# Test
# --------------------------------------------------
model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_preds = torch.argmax(test_logits, dim=1)
    test_acc = (test_preds == y_test).float().mean().item()

print("Best validation loss:", best_val_loss)
print("Test accuracy:", test_acc)

IndexError: Target 6 is out of bounds.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_preds = torch.argmax(test_logits, dim=1).cpu().numpy()
    y_test_numpy = y_test.cpu().numpy()

# 1. Confusion Matrix
cm = confusion_matrix(y_test_numpy, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# 2. Per-class Performance
print(classification_report(y_test_numpy, test_preds))

The model fails to classify activities 0, 1, and 2, resulting in a 0% recall for half of the dataset. This indicates that the LSTM is not learning discriminative temporal features and is instead defaulting to predicting Class 4 (which has 100% recall but very low precision) and Class 5. The high performance on Class 5 suggests it has a very distinct signal, while the confusion in other classes is likely caused by unnormalized input scales or a lack of mini-batch optimization.

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F

# Activity label map (0-indexed to string)
ACTIVITY_LABELS = {
    0: "WALKING",
    1: "WALKING_UPSTAIRS",
    2: "WALKING_DOWNSTAIRS",
    3: "SITTING",
    4: "STANDING",
    5: "LAYING",
}

# Per-channel normalization stats (fit on raw train data)
# Shape: (1, 1, 9) for broadcasting over (batch, T, C)
_mu  = X_train.mean(dim=(0, 1), keepdim=True)   # X_train is already a tensor
_std = X_train.std(dim=(0, 1),  keepdim=True) + 1e-8


def predict_activity(window):

    # --- Convert to tensor if needed ---
    if isinstance(window, np.ndarray):
        window = torch.tensor(window, dtype=torch.float32)

    # --- Normalize with training statistics ---
    window = window.to(device)          # (128, 9)
    window = (window - _mu[0]) / _std[0]

    # --- Add batch dimension → (1, 128, 9) ---
    x = window.unsqueeze(0)

    # --- Inference (no gradient, eval mode) ---
    model.eval()
    with torch.no_grad():
        logits = model(x)                           # (1, 6)
        probs  = F.softmax(logits, dim=-1).squeeze(0)  # (6,)

    probs_np  = probs.cpu().numpy()
    class_idx = int(probs_np.argmax())
    label     = ACTIVITY_LABELS[class_idx]

    return label, probs_np


# ── Quick sanity check ───────────────────────────────────────
# X_test here is the raw tensor before normalization.
# If you already normalized X_test in your training cell,
# replace X_test[0] with the raw numpy array for that window.
sample_window = X_test[0].cpu()   # shape (128, 9)
pred_label, pred_probs = predict_activity(sample_window)

print("=== Sanity Check ===")
print(f"True label : {ACTIVITY_LABELS[y_test[0].item()]}")
print(f"Predicted  : {pred_label}  (confidence {pred_probs.max()*100:.1f}%)\n")
print("Per-class probabilities:")
for idx, (name, p) in enumerate(zip(ACTIVITY_LABELS.values(), pred_probs)):
    bar = "█" * int(p * 40)
    print(f"  {name:<22} {p:.4f}  {bar}")


# ── Latency Benchmark ────────────────────────────────────────
N_BENCH = 500   # number of windows to time

# Force CPU for the benchmark (realistic for edge/mobile devices)
model_cpu = model.to("cpu")
_mu_cpu   = _mu.cpu()
_std_cpu  = _std.cpu()

def predict_activity_cpu(window):
    """CPU-only version for latency measurement."""
    if isinstance(window, np.ndarray):
        window = torch.tensor(window, dtype=torch.float32)
    window = window.cpu()
    window = (window - _mu_cpu[0]) / _std_cpu[0]
    x = window.unsqueeze(0)
    model_cpu.eval()
    with torch.no_grad():
        logits = model_cpu(x)
        probs  = F.softmax(logits, dim=-1).squeeze(0)
    probs_np  = probs.numpy()
    return ACTIVITY_LABELS[int(probs_np.argmax())], probs_np

# Warm-up run (avoids cold-start JIT effects)
for _ in range(10):
    predict_activity_cpu(X_test[0].cpu())

# Timed runs
latencies_ms = []
test_windows = X_test[:N_BENCH].cpu()

for i in range(N_BENCH):
    t0 = time.perf_counter()
    predict_activity_cpu(test_windows[i])
    latencies_ms.append((time.perf_counter() - t0) * 1000)

latencies_ms = np.array(latencies_ms)

print(f"\n=== Latency Benchmark (CPU, N={N_BENCH} windows) ===")
print(f"  Mean   : {latencies_ms.mean():.2f} ms/window")
print(f"  Median : {np.median(latencies_ms):.2f} ms/window")
print(f"  P95    : {np.percentile(latencies_ms, 95):.2f} ms/window")
print(f"  P99    : {np.percentile(latencies_ms, 99):.2f} ms/window")
print(f"  Min    : {latencies_ms.min():.2f} ms")
print(f"  Max    : {latencies_ms.max():.2f} ms")

WINDOW_DURATION_MS = 2560   # 128 samples @ 50 Hz
headroom_mean = WINDOW_DURATION_MS / latencies_ms.mean()
headroom_p99  = WINDOW_DURATION_MS / np.percentile(latencies_ms, 99)

print(f"\n=== Near-Real-Time Feasibility ===")
print(f"  Window duration : {WINDOW_DURATION_MS} ms  (128 samples @ 50 Hz)")
print(f"  Mean headroom   : {headroom_mean:.0f}× real-time")
print(f"  P99  headroom   : {headroom_p99:.0f}× real-time (worst-case)")
print(f"""
  Interpretation:
  The model's mean CPU inference time of {latencies_ms.mean():.1f} ms is well below the
  2,560 ms window budget, giving ~{headroom_mean:.0f}× real-time headroom on Colab's CPU.
  Even at the P99 worst case ({np.percentile(latencies_ms,99):.1f} ms), the model still
  processes windows ~{headroom_p99:.0f}× faster than they arrive. A practical
  near-real-time threshold for wearable/mobile pipelines is typically
  <100 ms per window; this model comfortably satisfies that constraint.
  On a weaker embedded CPU (e.g. Raspberry Pi or smartphone), latency
  would be higher, but the large headroom leaves substantial margin
  before real-time constraints are violated. Exporting to ONNX or
  TorchScript would reduce latency further by ~2–5× via kernel fusion.
""")

# Move model back to original device if needed
model.to(device)

=== Sanity Check ===
True label : LAYING
Predicted  : STANDING  (confidence 17.8%)

Per-class probabilities:
  WALKING                0.1659  ██████
  WALKING_UPSTAIRS       0.1508  ██████
  WALKING_DOWNSTAIRS     0.1568  ██████
  SITTING                0.1717  ██████
  STANDING               0.1779  ███████
  LAYING                 0.1769  ███████

=== Latency Benchmark (CPU, N=500 windows) ===
  Mean   : 9.56 ms/window
  Median : 6.31 ms/window
  P95    : 20.48 ms/window
  P99    : 25.59 ms/window
  Min    : 4.99 ms
  Max    : 27.91 ms

=== Near-Real-Time Feasibility ===
  Window duration : 2560 ms  (128 samples @ 50 Hz)
  Mean headroom   : 268× real-time
  P99  headroom   : 100× real-time (worst-case)

  Interpretation:
  The model's mean CPU inference time of 9.6 ms is well below the
  2,560 ms window budget, giving ~268× real-time headroom on Colab's CPU.
  Even at the P99 worst case (25.6 ms), the model still
  processes windows ~100× faster than they arrive. A practical
  near-r

LSTMClassifier(
  (lstm): LSTM(9, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)

In [ ]:
# ============================================================
# CHECKPOINT 15 — Model Card + Limitations
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║                        MODEL CARD                           ║
║                      HAR-LSTM  v1.0                         ║
╚══════════════════════════════════════════════════════════════╝

─── MODEL OVERVIEW ────────────────────────────────────────────
  Type       : 2-layer LSTM → Dropout → Linear classifier
  Input      : (batch, 128, 9)  — 9 IMU channels, 128 timesteps
  Output     : 6-class softmax (activity logits)
  Params     : ~2-layer LSTM hidden_dim=128, dropout=0.3
  Trained on : UCI HAR Dataset (waist-mounted smartphone IMU)

─── INTENDED USE ──────────────────────────────────────────────
  • Classify one of 6 physical activities in real time:
    WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS,
    SITTING, STANDING, LAYING
  • Target context: smartphone or wearable app using a
    waist-mounted IMU sensor at 50 Hz

─── NOT INTENDED FOR ──────────────────────────────────────────
  • Medical diagnosis or clinical use — not clinically validated
  • Fall detection or emergency alerts — false negatives are
    safety-critical and this model gives no uncertainty flag
  • Activities outside the 6 training classes (e.g. cycling,
    swimming) — model will still confidently output a wrong class
  • Populations not in the training data (children, elderly
    with mobility impairments, etc.)
  • Covert surveillance or employee monitoring

─── EVALUATION PROTOCOL ───────────────────────────────────────
  • Dataset   : UCI HAR (10,299 total windows)
  • Train     : 7,352 windows from 21 subjects
  • Test      : 2,947 windows from 9 held-out subjects
                (no subject overlap between train and test)
  • Metrics   : Accuracy, per-class Precision / Recall / F1
  • Scheduler : ReduceLROnPlateau (factor=0.5, patience=3)
  • Stopping  : Early stopping (patience=7)
  • Clipping  : Gradient norm clipped at 1.0

─── KNOWN FAILURE MODES ───────────────────────────────────────
  1. SITTING vs STANDING confusion — both are static, near-zero
     acceleration; hardest pair for any IMU-based model
  2. WALKING_UPSTAIRS vs DOWNSTAIRS — rhythmic step pattern is
     similar; vertical asymmetry is subtle across subjects
  3. Transition windows — a window that spans two activities
     gets a noisy label and will likely be misclassified
  4. Sensor placement — trained only on waist-worn data;
     accuracy drops sharply if phone is in hand or a bag
  5. Out-of-distribution activities — any activity not in the
     6 classes is silently mapped to the nearest class

─── PRIVACY & ETHICS NOTE ─────────────────────────────────────
  IMU data can reveal daily routines and has been shown to
  allow re-identification of individuals via gait patterns.
  Any system using this model should:
    • Obtain informed user consent before continuous monitoring
    • Store raw sensor data locally; transmit only activity labels
    • Be transparent with users about what is being inferred
    • Not infer sensitive attributes (health, religion, routine)
      without explicit consent
""")


╔══════════════════════════════════════════════════════════════╗
║                        MODEL CARD                           ║
║                      HAR-LSTM  v1.0                         ║
╚══════════════════════════════════════════════════════════════╝

─── MODEL OVERVIEW ────────────────────────────────────────────
  Type       : 2-layer LSTM → Dropout → Linear classifier
  Input      : (batch, 128, 9)  — 9 IMU channels, 128 timesteps
  Output     : 6-class softmax (activity logits)
  Params     : ~2-layer LSTM hidden_dim=128, dropout=0.3
  Trained on : UCI HAR Dataset (waist-mounted smartphone IMU)

─── INTENDED USE ──────────────────────────────────────────────
  • Classify one of 6 physical activities in real time:
    WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS,
    SITTING, STANDING, LAYING
  • Target context: smartphone or wearable app using a
    waist-mounted IMU sensor at 50 Hz

─── NOT INTENDED FOR ──────────────────────────────────────────
  • Medical diagnosis or c

In [ ]:
# Interpretability: saliency/gradient-based attribution over timesteps/channels with a short discussion of a failure mode.
import torch
import matplotlib.pyplot as plt

def compute_saliency(model, x_sample, target_class=None):
    """
    x_sample: shape (1, T, C)
    target_class: optional integer class index
    returns:
        saliency: absolute gradient, shape (T, C)
        pred_class: predicted class
    """
    model.eval()

    x_sample = x_sample.clone().detach().requires_grad_(True)

    logits = model(x_sample)   # shape: (1, num_classes)
    pred_class = torch.argmax(logits, dim=1).item()

    if target_class is None:
        target_class = pred_class

    score = logits[0, target_class]

    model.zero_grad()
    score.backward()

    saliency = x_sample.grad.detach().abs().squeeze(0)   # (T, C)

    return saliency, pred_class


# pick one test example
idx = 0
x_sample = X_test[idx:idx+1].to(device)

saliency, pred_class = compute_saliency(model, x_sample)

print("Predicted class:", pred_class)
print("Saliency shape:", saliency.shape)   # (T, C)


timestep_saliency = saliency.mean(dim=1).cpu().numpy()

plt.figure(figsize=(10, 4))
plt.plot(timestep_saliency)
plt.title("Saliency over Timesteps")
plt.xlabel("Timestep")
plt.ylabel("Mean |gradient| across channels")
plt.show()


channel_saliency = saliency.mean(dim=0).cpu().numpy()
plt.figure(figsize=(8, 4))
plt.bar(range(len(channel_saliency)), channel_saliency)
plt.title("Saliency over Channels")
plt.xlabel("Channel")
plt.ylabel("Mean |gradient| across timesteps")
plt.show()


idx = 0
x_sample = X_test[idx:idx+1].to(device)
true_class = y_test[idx].item()

saliency, _ = compute_saliency(model, x_sample, target_class=true_class)


"""
Gradient-based saliency was used to examine which timesteps and channels most influenced
the LSTM’s prediction. The timestep plot shows that attribution is very small for most of
the sequence and rises sharply near the end, indicating that the model relies mainly on
the last part of the input window. The channel plot shows uneven feature usage, with channels
6, 7, and 8 contributing the most, especially channel 6. This suggests the model focuses on a
small subset of input features. A possible failure mode is that saliency can be noisy and may
overemphasize locally sensitive regions rather than truly meaningful temporal structure.
"""